In [1]:
import requests
import json
from neo4j import GraphDatabase


In [2]:
OLLAMA_URL     = "http://localhost:11434/api/generate"
MODEL          = "gemma3:4b"
NEO4J_URI      = "neo4j://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "123456789"

In [3]:
SCHEMA = """
Graph schema:
- (Document {id, text_preview})
- (Entity   {text, label})   labels: PERSON, ORG, GPE, DATE, MONEY, LOC, MISC
- (Token    {text, pos})     pos: NN, NNP, VB, JJ, ...
- (Document)-[:HAS_ENTITY {start, end}]->(Entity)
- (Document)-[:HAS_TOKEN]->(Token)
- (Entity)-[:CO_OCCURS_WITH {doc_id}]->(Entity)
"""

CYPHER_PROMPT = """You are a Neo4j Cypher expert. Convert the user question to a Cypher query.

{schema}

Rules:
- Return ONLY the Cypher query, nothing else
- No markdown, no explanation, no code fences
- Use LIMIT 20 unless the question asks for counts
- For name searches use case-insensitive: toLower(e.text) CONTAINS toLower($value)

Question: {question}

Cypher:"""

ANSWER_PROMPT = """You are a helpful assistant. Answer the user's question based on the database results.

Question: {question}

Database results:
{results}

Give a clear, concise answer in the same language as the question.
If results are empty, say no data was found."""


In [4]:
def call_ollama(prompt: str, max_tokens: int = 500) -> str:
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": max_tokens}
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except Exception as e:
        return f"ERROR: {e}"


def run_cypher(query: str) -> list:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    try:
        with driver.session() as session:
            result = session.run(query)
            return [dict(record) for record in result]
    except Exception as e:
        return [{"error": str(e)}]
    finally:
        driver.close()


def clean_cypher(raw: str) -> str:
    """شيل أي حاجة زيادة من الـ Cypher."""
    raw = raw.strip()
    # شيل code fences
    for fence in ["```cypher", "```", "`"]:
        raw = raw.replace(fence, "")
    return raw.strip()


In [5]:
def chat(question: str) -> str:
    # Step 1: ولّد الـ Cypher
    cypher_raw = call_ollama(CYPHER_PROMPT.format(schema=SCHEMA, question=question))
    cypher = clean_cypher(cypher_raw)

    results = run_cypher(cypher)

    if results and "error" in results[0]:
        return f"❌ خطأ في الـ query: {results[0]['error']}"

    results_str = json.dumps(results[:10], ensure_ascii=False, indent=2)
    answer = call_ollama(
        ANSWER_PROMPT.format(question=question, results=results_str),
        max_tokens=800
    )
    return answer

In [6]:
"Who are the people mentioned in the documents?"

'Who are the people mentioned in the documents?'

In [8]:
question = "Who are the people mentioned in the documents?"
print(chat(question))

Who are the people mentioned in the documents?

Charles Dickens, Richard the Second, Mr Krook, Countess Cornelia de Baudi Cesenate, Giuseppe Bianchini, Le Cat, Lord High Chancellor, and solicitors.
